# 9주차. 검증 모델과 쿼리 파라미터 모델링

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |

이 노트북은 수업 주차 흐름을 참고해 우리 방식으로 재구성한 실행형 설명 자료입니다. 외부 서비스 호출 없이 실행되도록 작은 샘플 데이터를 사용합니다.

## 검증 모델
요청 파라미터가 많아지면 하나의 모델로 묶어 관리하는 편이 안전합니다. 여기서는 dataclass로 모델의 핵심 원리를 표현합니다.

In [1]:
from dataclasses import dataclass
import pandas as pd

@dataclass
class FilterParams:
    q: str | None = None
    limit: int = 10
    offset: int = 0
    order_by: str = "id"

    def validate(self):
        if not 1 <= self.limit <= 100:
            raise ValueError("limit must be 1..100")
        if self.offset < 0:
            raise ValueError("offset must be non-negative")
        if self.order_by not in {"id", "name", "price"}:
            raise ValueError("unsupported order field")
        return self

rows = pd.DataFrame([
    {"id": 1, "name": "api", "price": 20},
    {"id": 2, "name": "jinja", "price": 25},
    {"id": 3, "name": "sqlite", "price": 15},
])

def search_items(params: FilterParams):
    params.validate()
    result = rows.copy()
    if params.q:
        result = result[result["name"].str.contains(params.q, case=False)]
    result = result.sort_values(params.order_by).iloc[params.offset: params.offset + params.limit]
    return result

result = search_items(FilterParams(q="i", limit=2, order_by="price"))
assert len(result) == 2
result

,id,name,price
2,3,sqlite,15
0,1,api,20
